# Построение датасета (dataset) из истории расчётов нагрузки

Этот ноутбук загружает CSV‑выгрузки (CSV exports) из таблицы ``,
фильтрует только **согласованные, нормируемые** записи, разбирает JSON‑поля (JSON fields)
и сохраняет плоский датасет (flat dataset) `operations_from_history.csv` в `data/processed/`.

Перед запуском убедитесь, что файлы `data-2022.csv`, `data-2023.csv`, `data-2024.csv`, `data-2025.csv`
лежат в папке `data/raw/` проекта.


In [ ]:
import pandas as pd
import json
from pathlib import Path

# Папка с сырыми CSV‑выгрузками (CSV exports)
RAW_DIR = Path("data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

# Папка для сохранения подготовленных датасетов (processed datasets)
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES


In [ ]:
# Загрузка всех CSV в единый датафрейм (single dataframe)

dfs = []
for path in RAW_FILES:
    print(f"Читаю {path} ...")
    df_part = pd.read_csv(path, dtype=str)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

# Преобразуем период (period) к типу дата (datetime)
df["period"] = pd.to_datetime(df["period"])

print("Всего строк:", len(df))
df.head()


In [ ]:
# Нормализация булевых полей (boolean fields)

def to_bool(series: pd.Series) -> pd.Series:
    return series.str.lower().isin(["true", "t", "1"])

for col in ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]:
    df[col] = to_bool(df[col])

# Количество этапов согласования (validation stages)
df["validate_count"] = (
    df["validate_count"].astype(float).fillna(0).astype(int)
)

# Фильтр: согласованные и нормируемые записи (approved & forecast)
mask_await_send = ~df["send"]
mask_cancelled = df["cancel_approve"]
mask_wait_second = (
    (df["validate_count"] == 2)
    & df["validate_one"]
    & ~df["validate_two"]
)

mask_not_approved = mask_await_send | mask_cancelled | mask_wait_second

df_ok = df[~mask_not_approved & df["forecast"]].copy()

print("Строк после фильтра:", len(df_ok))
df_ok.head()


In [ ]:
# Разбор JSON с данными о подразделении (division_name JSON)

def parse_division_name(s: str) -> pd.Series:
    if pd.isna(s):
        return pd.Series({"division_id": None, "division_title": None})
    try:
        obj = json.loads(s)
    except Exception:
        return pd.Series({"division_id": None, "division_title": None})
    return pd.Series({
        "division_id": obj.get("id"),
        "division_title": obj.get("name"),
    })

div_cols = df_ok["division_name"].apply(parse_division_name)
df_ok = pd.concat([df_ok, div_cols], axis=1)

df_ok[["resist_id", "division_id", "division_title", "period"]].head()


In [ ]:
# Разбор JSON с операциями (operation_parameter JSON)

def extract_operations_from_row(row, value_key="value"):
    raw = row.get("operation_parameter")
    if pd.isna(raw):
        return []
    try:
        data = json.loads(raw)
    except Exception:
        return []

    records = []
    for op_id, payload in data.items():
        if isinstance(payload, dict):
            op_value = payload.get(value_key)
        else:
            op_value = payload
        records.append({
            "history_id": row["id"],
            "resist_id": row["resist_id"],
            "division_id": row.get("division_id"),
            "division_title": row.get("division_title"),
            "period": row["period"],
            "operation_id": op_id,
            "operation_value": op_value,
        })
    return records

all_records = []
for _, r in df_ok.iterrows():
    all_records.extend(extract_operations_from_row(r))

df_ops = pd.DataFrame(all_records)

df_ops["operation_value"] = pd.to_numeric(df_ops["operation_value"], errors="coerce").fillna(0)
df_ops["period"] = pd.to_datetime(df_ops["period"])

print("Строк в таблице операций:", len(df_ops))
df_ops.head()


In [ ]:
# Сохранение подготовленного датасета (processed dataset)

out_path = PROCESSED_DIR / "operations_from_history.csv"
df_ops.to_csv(out_path, index=False)

print("Файл сохранён в:", out_path)
